In [1]:
"""Split a patent sheet into its separate figures.

Rule of thumb: never merge two things just because their bounding *boxes* touch.

  1. work on the ink (connected components), not on filled rectangles
  2. big components are figure "seeds"
  3. a smaller piece (label letter, arrow, stray stroke) joins a seed only if
     most of *its own* box lies inside the seed's box
"""
import cv2 as cv
import numpy as np


def area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


def frac_inside(piece, group):
    """Share of `piece`'s box lying inside `group`'s box (overlap / smaller area, not IoU)."""
    iw = min(piece[2], group[2]) - max(piece[0], group[0])
    ih = min(piece[3], group[3]) - max(piece[1], group[1])
    return max(iw, 0) * max(ih, 0) / area(piece)


MIN_AREA  = 20    # ink pixels; drops speckles
CLOSE_PX  = 0     # >1: dilate the ink first to bridge broken strokes / tiny gaps.
                  # Keep it well BELOW the gap between two figures. On the sample page
                  # (gap ~9.5 px) Fig 1 / Fig 2 start to leak at 9 and fuse at 11.
CONTAIN   = 0.5   # a piece joins a figure if >= 50% of its box is inside the figure's box
FIG_FRAC  = 0.02  # groups covering >= 2% of the page count as figures


def find_figures(ink, min_area=MIN_AREA, close_px=CLOSE_PX, contain=CONTAIN):
    """ink: uint8 image, ink = 255. Returns a list of [x1, y1, x2, y2] (x2, y2 exclusive)."""
    work = cv.dilate(ink, np.ones((close_px, close_px), np.uint8)) if close_px > 1 else ink
    _, _, stats, _ = cv.connectedComponentsWithStats(work, connectivity=8)
    parts = [(int(x), int(y), int(x + w), int(y + h))
             for x, y, w, h, a in stats[1:] if a >= min_area]

    groups = []
    for p in sorted(parts, key=area, reverse=True):          # biggest first = seeds
        host = max(groups, key=lambda g: frac_inside(p, g), default=None)
        if host is not None and frac_inside(p, host) >= contain:
            host[:] = [min(host[0], p[0]), min(host[1], p[1]),
                       max(host[2], p[2]), max(host[3], p[3])]
        else:
            groups.append(list(p))
    return groups



In [6]:
import glob
from pathlib import Path as PATH
import os

def ink_based_connected_components(src, debug_out):
    gray = cv.imread(src, cv.IMREAD_GRAYSCALE)
    _, ink = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)
    H, W = ink.shape
    figures = sorted((g for g in find_figures(ink) if area(g) >= FIG_FRAC * W * H),
                     key=lambda b: (b[1], b[0]))            # reading order
    debug = cv.cvtColor(ink, cv.COLOR_GRAY2BGR)
    for i, (x1, y1, x2, y2) in enumerate(figures, 1):
        cv.rectangle(debug, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 0), 2)   # corners are inclusive
    # uncomment the following lines to save each figure as a separate image
    #     cv.imwrite(f"figure_{i}.png", gray[y1:y2, x1:x2])
    cv.imwrite(debug_out, debug)
    # print(f"{len(figures)} figures:", figures)


directory_path = "../../data/figure_extractor_test/"
all_sketch_pages = glob.glob(directory_path + "*.png")
debug_directory = PATH("../../results/figure_extractor_test_output/")
debug_directory.mkdir(exist_ok=True)

for src in all_sketch_pages:
    src_filename = src.split("/")[-1]
    debug_out = os.path.join(debug_directory, PATH(f"debug_{src_filename}"))
    print(f"Debug output saved to: {debug_out}")
    ink_based_connected_components(src, debug_out)

Debug output saved to: ../../results/figure_extractor_test_output/debug_2393_2398_page_3.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2424_2429_page_2.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2424_2429_page_3.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2398_2403_page_3.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2398_2403_page_4.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2453_2458_page_3.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2394_2399_page_2.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2451_2456_page_2.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2451_2456_page_3.png
Debug output saved to: ../../results/figure_extractor_test_output/debug_2382_2387_page_2.png
Debug output saved to: ../../results/figure_extractor_test_output/debu